# SAHI COCO Utilities

SAHI includes a comprehensive set of tools for working with COCO-format datasets:
- **`slice_coco()`**: Slice a large annotated COCO dataset into smaller training patches
- **`calculate_stats()`**: Compute detection evaluation metrics (AP, AR)
- **`export_coco_as_yolov5()`**: Convert to YOLO training format

These utilities are especially useful when preparing remote sensing datasets where objects are small relative to the scene size.

## References
- SAHI COCO utils: https://github.com/obss/sahi/blob/main/sahi/utils/coco.py
- COCO format spec: https://cocodataset.org/#format-data

## 1. Download a Sample COCO Dataset

In [ ]:
import os
import urllib.request
import json

# Sample COCO annotation file from SAHI's test fixtures
ann_url = "https://raw.githubusercontent.com/obss/sahi/main/tests/data/coco_utils/terrain2_coco.json"
img_url = "https://raw.githubusercontent.com/obss/sahi/main/demo/demo_data/coco_utils/terrain2_coco.jpg"

os.makedirs("coco_sample/images", exist_ok=True)
if not os.path.exists("coco_sample/annotations.json"):
    urllib.request.urlretrieve(ann_url, "coco_sample/annotations.json")
if not os.path.exists("coco_sample/images/terrain2_coco.jpg"):
    urllib.request.urlretrieve(img_url, "coco_sample/images/terrain2_coco.jpg")

with open("coco_sample/annotations.json") as f:
    coco = json.load(f)

print(f"Images: {len(coco['images'])}")
print(f"Annotations: {len(coco['annotations'])}")
print(f"Categories: {[c['name'] for c in coco['categories']]}")

## 2. Slice the Dataset for Training

`slice_coco()` tiles the images and adjusts bounding box annotations to match each tile's coordinate space. Partial boxes that span tile edges are clipped and retained.

In [ ]:
from sahi.slicing import slice_coco

coco_dict, coco_path = slice_coco(
    coco_annotation_file_path="coco_sample/annotations.json",
    image_dir="coco_sample/images/",
    output_coco_annotation_file_name="sliced_annotations",
    output_dir="coco_sliced/",
    slice_height=320,
    slice_width=320,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    min_area_ratio=0.1,  # discard annotations covering <10% of a tile
    verbose=True,
)

print(f"\nOriginal annotations: {len(coco['annotations'])}")
print(f"Sliced annotations: {len(coco_dict['annotations'])}")
print(f"Sliced images: {len(coco_dict['images'])}")

## 3. Visualize Sliced Tiles

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches

# Show first 4 tiles with their annotations
sample_images = coco_dict["images"][:4]
ann_by_image = {}
for ann in coco_dict["annotations"]:
    ann_by_image.setdefault(ann["image_id"], []).append(ann)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, img_info in zip(axes, sample_images):
    img_path = os.path.join("coco_sliced", img_info["file_name"])
    if os.path.exists(img_path):
        ax.imshow(mpimg.imread(img_path))
        for ann in ann_by_image.get(img_info["id"], []):
            x, y, w, h = ann["bbox"]
            rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor="red", facecolor="none")
            ax.add_patch(rect)
    ax.set_title(f"tile {img_info['id']}")
    ax.axis("off")
plt.suptitle("Sample sliced training tiles with annotations")
plt.tight_layout()
plt.show()

## 4. Export to YOLO Format

In [ ]:
from sahi.utils.coco import Coco

coco_obj = Coco.from_coco_dict_or_path(
    coco_path,
    image_dir="coco_sliced/",
)

coco_obj.export_as_yolov5(
    output_dir="yolo_dataset/",
    train_split_rate=0.85,
)
print("Exported YOLO dataset to yolo_dataset/")
print("Structure: yolo_dataset/train/images/, yolo_dataset/val/images/")

## 5. Evaluate Detections Against Ground Truth

`calculate_stats()` computes COCO-standard AP/AR metrics between a ground truth annotation file and a set of predictions.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.coco import Coco

model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="yolov8n.pt",
    confidence_threshold=0.25,
    device="cpu",
)

# Run predictions on all sliced images and save as COCO result format
result = get_sliced_prediction(
    image="coco_sample/images/terrain2_coco.jpg",
    detection_model=model,
    slice_height=320,
    slice_width=320,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    verbose=0,
)

print(f"Predicted {len(result.object_prediction_list)} objects")
print("\nNote: Full AP/AR evaluation requires matching image IDs between")
print("predictions and ground truth. See sahi.utils.coco.calculate_stats()")
print("for usage with evaluation datasets.")